# Satellite Scheduling — Visibility Windows → CSP → Dimensionality Reduction

Full pipeline from wrangled data to a problem whose complexity has been reduced using CSP.

In [1]:
import sys, copy, time
import json
from itertools import combinations
from pathlib import Path

import pandas as pd


def find_project_root(start: Path | None = None) -> Path:
    """
    Find the repository root so paths work without further trouble.
    """
    current = (start or Path.cwd()).resolve()
    for candidate in [current, *current.parents]:
        if (candidate / "data").exists() and (candidate / "results").exists():
            return candidate
    raise FileNotFoundError("Could not find project root containing both `data/` and `results/`.")

# loading project root dir
PROJECT_ROOT = find_project_root()
RESULTS_DIR = PROJECT_ROOT / "results"

# project modules live in `src/`
sys.path.insert(0, str(PROJECT_ROOT / "src"))
from CSP import CSP, CSP_Variable, Timeslot, ac3, split_connected_components


# --------- loading configuration environment ----------
# load .json config file (same file used by the MWIS and QUBO notebooks)
CONFIG_PATH = RESULTS_DIR / "_config.json"
run_config = json.loads(CONFIG_PATH.read_text(encoding="utf-8"))

# input: candidate slots produced by the data-wrangling notebooks
CSP_SLOT_CSV = PROJECT_ROOT / run_config["csp_slot_csv"]

# output: artifacts of this stage
RUN_NAME = run_config["run_name_csp"]
RUN_DIR = RESULTS_DIR / "runs" / "csp_showcase" / RUN_NAME
RUN_DIR.mkdir(parents=True, exist_ok=True)

# loggin environment configuration
(RUN_DIR / "run_config_log.json").write_text(json.dumps(run_config, indent=2), encoding="utf-8")

print(f"Loaded configuration from: {CONFIG_PATH.resolve()}")
print(f"Artifacts will be saved to: {RUN_DIR.resolve()}")

Loaded configuration from: D:\University\PoliMi\courses\SWE2\SWE2-research-project\results\_config.json
Artifacts will be saved to: D:\University\PoliMi\courses\SWE2\SWE2-research-project\results\runs\csp_showcase\latest_csp_run


## 1. Load the Scheduling Instance

In [2]:
# dataset chosen in `_config.json` (switch between small / medium / dense for different problem scales)
df = pd.read_csv(CSP_SLOT_CSV)

# the discretisation step is the distance between two consecutive candidate slots:
# it is the time unit the CSP works with, so times become integer indexes
disc_step_s = int(df['discretization_step_seconds'].iloc[0])
disc_step_h = disc_step_s / 3600

print("-"*10 + " dataset summary info " + "-"*10)
print(f'Dataset        : {CSP_SLOT_CSV.name}')
print(f'Num. candidate slots : {len(df):,}')
print(f'Num. satellites      : {df["satellite_name"].nunique()}')
print(f'Num. antennas        : {df["antenna_id"].nunique()}')
print(f'Num. parent windows  : {df["parent_window_id"].nunique()}')
print(f'Num. disc. step      : {disc_step_s}s  ({disc_step_h:.4f}h)')
print(f'Tot. time horizon    : {df["start_time_hours"].min():.3f} - {df["start_time_hours"].max():.3f}h')

df[['slot_id', 'satellite_name', 'antenna_id', 'parent_window_id',
    'start_time_hours', 'end_time_hours', 'candidate_index_in_parent',
    'priority', 'urgency', 'weight']].head(5)

---------- dataset summary info ----------
Dataset        : starlink_scheduling_slots_weighted.csv
Num. candidate slots : 2,556
Num. satellites      : 95
Num. antennas        : 50
Num. parent windows  : 2005
Num. disc. step      : 120s  (0.0333h)
Tot. time horizon    : 0.000 - 3.917h


,slot_id,satellite_name,antenna_id,parent_window_id,start_time_hours,end_time_hours,candidate_index_in_parent,priority,urgency,weight
0,SLOT_000001,STARLINK-1296,ANT_GATEWAY_0001,VIS_000001,0.0,0.083333,1,2,5,56.295
1,SLOT_000002,STARLINK-1269,ANT_GATEWAY_0004,VIS_000007,0.0,0.083333,1,1,5,39.304
2,SLOT_000003,STARLINK-1263,ANT_GATEWAY_0005,VIS_000013,0.0,0.083333,1,2,5,52.784
3,SLOT_000004,STARLINK-1269,ANT_GATEWAY_0007,VIS_000017,0.0,0.083333,1,2,5,59.100
4,SLOT_000005,STARLINK-1350,ANT_GATEWAY_0007,VIS_000018,0.0,0.083333,1,5,5,83.618


## 2. Time-Discrete Reference System

Each raw *visibility window* says that *Satellite **S** can communicate with antenna **A** between **T₁** and **T₂**.*

**Discretisation** slides a fixed-length contact window of `min_contact_duration` over \[T₁, T₂\] in steps of `disc_step_s`, generating one *candidate slot* per valid placement. Each parent window therefore yields 1–N alternatives, all with the same priority/urgency inherited from the parent.

## 3. Translating to CSP

| Entity | CSP element |
|---|---|
| Satellite *S* | `CSP_Variable` (one per unique satellite) |
| Candidate slot at antenna *A*, starting at time *T* | `Timeslot(antenna_id=int(A), timeframe=round(T / disc_step_h))` |
| All candidate slots of satellite *S* | Domain of variable *S* |
| Two satellites share a `Timeslot` | *Constraint* (Diff formulation: they cannot both be assigned it) |

In [3]:
# TODO: shape this part passing by the model class

satellites = sorted(df['satellite_name'].unique())
antennas   = sorted(df['antenna_id'].unique())
# look-up dictionary to index variables
# (names are sorted, so the same dataset always yields the same ids)
sat_to_id  = {s: i for i, s in enumerate(satellites)}
ant_to_id  = {a: i for i, a in enumerate(antennas)}

# to convert continous the staring time in some kind of discrete index
# two slots sharing (antenna, timeframe) are the *same* resource: this is what creates conflicts
def to_timeframe(start_h: float) -> int:
    return round(start_h / disc_step_h)

# build the CSP: one variable per satellite, its domain is the slots it may use
csp = CSP(max_vars=len(satellites))

for sat_name, sat_rows in df.groupby('satellite_name', sort=True):
    # every slot of a satellite inherits the same weight from its parent window
    var = CSP_Variable(id=sat_to_id[sat_name], name=sat_name, weight=sat_rows['weight'].iloc[0])
    for row in sat_rows.itertuples(index=False):
        var.add_value(Timeslot(
            antenna_id=ant_to_id[row.antenna_id],
            timeframe=to_timeframe(row.start_time_hours),
        ))
    # constraints are implicit: add_var links variables sharing a Timeslot (Diff formulation)
    csp.add_var(var)


# --- prettier summary print ---
def print_summary(label, summ):
    print(f'\n{"-"*46}')
    print(f'  {label}')
    print(f'{"-"*46}')
    for k, v in summ.items():
        val = f'{v:.4f}' if isinstance(v, float) else str(int(v))
        print(f'  {k:<24} {val}')

summary_before = csp.compute_summary()
print_summary('CSP  —  BEFORE preprocessing', summary_before)

# AC-3 prunes domains in place, so keep an untouched copy to compare against
csp_before = copy.deepcopy(csp)


----------------------------------------------
  CSP  —  BEFORE preprocessing
----------------------------------------------
  num_vars                 95
  avg_domain_size          25.8737
  sd_domain_size           16.4408
  min_domain_size          1.0000
  max_domain_size          74.0000
  num_const                234
  avg_degree               4.9263
  sd_degree                2.5884
  const_dens               0.0524


## 4. Dimensionality Reduction
The **Diff formulation** add a constraint between two satellites if their domains share a `Timeslot`. 

The problem's complexity decomposition is composed by two phases:

### AC-3 Arc Consistency

AC-3 iteratively:
1. Removes unsupported values from domains
2. Prunes constraints between variables whose domains have become disjoint
3. Propagates changes to neighbours

Returns `False` if any domain is wiped out (infeasible problem).

### Connected-Component Decomposition

After AC-3, variables with no remaining shared slots are completely independent. Splitting the CSP into connected components allows each sub-problem to be sent separately to the quantum annealer — dramatically reducing the QUBO size per call.

In [4]:
# AC-3 enforces arc consistency: it removes from every domain the slots that
# cannot appear in any solution. It shrinks the problem without cutting solutions;
# an empty domain means the instance has none at all.
t0 = time.time()
feasible = ac3(csp)
elapsed_ms = (time.time() - t0) * 1000

if not feasible: print('AC-3: problem turned out to be unfeasible')
else:
    print(f'AC-3 completed in {elapsed_ms:.1f} ms')

summary_after_ac3 = csp.compute_summary()
print_summary('CSP  —  AFTER AC-3', summary_after_ac3)

# how much did AC-3 actually reduce? (fewer domain values -> fewer QUBO variables downstream)
pruned = [
    (v.name, len(csp_before._var_by_id[v.id].domain), len(v.domain))
    for v in csp.vars
    if len(v.domain) < len(csp_before._var_by_id[v.id].domain)
]
print(f'\nSatellites with pruned domains : {len(pruned)}')
print(f'Constraints removed by AC-3   : {int(summary_before["num_const"] - summary_after_ac3["num_const"])}')
if pruned:
    print('\nPruned variables (name, before, after):')
    for name, b, a in pruned:
        print(f'  {name:<20}  {b} -> {a}  (-{b-a})')

AC-3: problem turned out to be unfeasible

----------------------------------------------
  CSP  —  AFTER AC-3
----------------------------------------------
  num_vars                 95
  avg_domain_size          25.8632
  sd_domain_size           16.4570
  min_domain_size          0.0000
  max_domain_size          74.0000
  num_const                234
  avg_degree               4.9263
  sd_degree                2.5884
  const_dens               0.0524

Satellites with pruned domains : 1
Constraints removed by AC-3   : 0

Pruned variables (name, before, after):
  STARLINK-1090         1 -> 0  (-1)


In [5]:
# satellites that share no timeslot can never conflict: the constraint graph splits
# into independent sub-problems, each solvable on its own and merged back at the end
con_components = split_connected_components(csp)
sizes = sorted([len(c.vars) for c in con_components], reverse=True)

print(f'Connected components after AC-3 : {len(con_components)}')
print(f'  Sizes (# satellites)  : {sizes}')
print(f'  Largest component     : {sizes[0]} satellites')
print(f'  Singleton components  : {sum(1 for s in sizes if s == 1)}')

Connected components after AC-3 : 10
  Sizes (# satellites)  : [85, 2, 1, 1, 1, 1, 1, 1, 1, 1]
  Largest component     : 85 satellites
  Singleton components  : 8


In [6]:
# ----------------------------------------------------------
# Export the CSP structure for the graphics notebook
# ----------------------------------------------------------

# one row per satellite: domain and degree, before and after AC-3
variables_rows = []
for before in csp_before.vars:
    after = csp._var_by_id[before.id]
    variables_rows.append({
        'var_id': before.id,
        'name': before.name,
        'weight': before.weight,
        'domain_size_before': len(before.domain),
        'domain_size_after': len(after.domain),
        'degree_before': csp_before.degree(before),
        'degree_after': csp.degree(after),
    })
pd.DataFrame(variables_rows).to_csv(RUN_DIR / 'csp_variables.csv', index=False)

# one row per constraint (satellite pair sharing a timeslot), for both stages
constraint_rows = [
    {'stage': stage, 'u_id': v.id, 'v_id': u.id}
    for stage, problem in (('before', csp_before), ('after', csp))
    for v in problem.vars
    for u in problem.neighbors(v)
    if u.id > v.id
]
pd.DataFrame(constraint_rows).to_csv(RUN_DIR / 'csp_constraints.csv', index=False)

# one row per connected component found after AC-3 (largest first)
sorted_components = sorted(con_components, key=lambda c: len(c.vars), reverse=True)
components_rows = [
    {
        'component_id': f'component_{index:03d}',
        'num_vars': len(component.vars),
        'num_constraints': component.num_constraints(),
        'qubo_vars': sum(len(v.domain) for v in component.vars),
    }
    for index, component in enumerate(sorted_components, start=1)
]
pd.DataFrame(components_rows).to_csv(RUN_DIR / 'csp_components.csv', index=False)

print(f'Exported CSP structure to: {RUN_DIR.resolve()}')

Exported CSP structure to: D:\University\PoliMi\courses\SWE2\SWE2-research-project\results\runs\csp_showcase\latest_csp_run


## 5. Hand-over to the QUBO notebook

Each sub-problem is exported as a **conflict graph**, the format `3_qubo_annealing.ipynb` reads: one node per `(satellite, timeslot)` pair, weighted by the satellite's weight, and one edge per pair that cannot both be selected.

| CSP element | Conflict graph |
|---|---|
| Binary variable `x[v, t]` | Node |
| Reward for scheduling a satellite | Node weight |
| One-hot penalty (one slot per satellite) | Edge between the nodes of the same satellite |
| Conflict penalty (one satellite per timeslot) | Edge between the nodes sharing a timeslot |

An independent set of this graph is exactly a feasible assignment, so the QUBO built downstream is the same one this notebook would build itself.

In [7]:
# ----------------------------------------------------------
# Export the sub-problems in the format read by 3_qubo_annealing.ipynb
# ----------------------------------------------------------

def component_to_conflict_graph(component) -> dict:
    """Turn one CSP component into a weighted conflict graph (nodes + edges as plain dicts)."""
    nodes, edges = [], []
    slot_users = {}     # (antenna, timeframe) -> nodes competing for that slot

    for v in component.vars:
        var_nodes = []
        for t in v.domain:
            node_id = f'v{v.id}_a{t.antenna_id}_t{t.timeframe}'
            nodes.append({
                'id': node_id,
                'satellite': v.name,
                'var_id': v.id,
                'antenna_id': t.antenna_id,
                'timeframe': t.timeframe,
                'weight': float(v.weight),
            })
            var_nodes.append(node_id)
            slot_users.setdefault((t.antenna_id, t.timeframe), []).append(node_id)

        # one-hot: a satellite takes at most one of its own slots
        edges += [{'source': u, 'target': w, 'reason': 'same satellite'}
                  for u, w in combinations(var_nodes, 2)]

    # conflict: a timeslot serves at most one satellite
    for users in slot_users.values():
        edges += [{'source': u, 'target': w, 'reason': 'same timeslot'}
                  for u, w in combinations(users, 2)]

    return {'nodes': nodes, 'edges': edges}


terminal_subgraphs = []
for index, component in enumerate(sorted_components, start=1):
    component_id = f'component_{index:03d}'
    graph = component_to_conflict_graph(component)
    terminal_subgraphs.append({
        'terminal_id': component_id,
        'original_component_id': component_id,
        'node_count': len(graph['nodes']),
        'edge_count': len(graph['edges']),
        'graph': graph,
    })

(RUN_DIR / 'terminal_subgraphs.json').write_text(json.dumps(terminal_subgraphs, indent=2), encoding='utf-8')

print(f'Exported {len(terminal_subgraphs)} sub-problems to: {(RUN_DIR / "terminal_subgraphs.json").resolve()}')
print(f'  Largest sub-problem : {max(t["node_count"] for t in terminal_subgraphs):,} nodes, '
      f'{max(t["edge_count"] for t in terminal_subgraphs):,} couplers')
print(f'  QPU limits in _config.json : {run_config["max_qpu_terminal_nodes"]} nodes, '
      f'{run_config["max_qpu_terminal_couplers"]} couplers (components are NOT decomposed yet)')

Exported 10 sub-problems to: D:\University\PoliMi\courses\SWE2\SWE2-research-project\results\runs\csp_showcase\latest_csp_run\terminal_subgraphs.json
  Largest sub-problem : 2,438 nodes, 44,305 couplers
  QPU limits in _config.json : 30 nodes, 200 couplers (components are NOT decomposed yet)
